## Vector store and retrive data

In [1]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and frienliness",
        metadata={"source":"mammal-pets-doc"},

    ),

     Document(
        page_content="Cats are independent pets that often enjoy thier own space",
        metadata={"source":"mammal-pets-doc"},

    ),

     Document(
        page_content="Goldfish are populer pets for beginners, requiring relatively simple care.",
        metadata={"source":"mammal-pets-doc"},

    ),

     Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source":"mammal-pets-doc"},

    )




]

documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and frienliness'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy thier own space'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Goldfish are populer pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

llm=ChatGroq(groq_api_key=groq_api_key,model="llama-3.1-8b-instant")

llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000018011D0DB20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000018011BE64E0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


# vectores

In [11]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents,embedding=embeddings)
vectorstore


In [12]:
vectorstore.similarity_search("cat")

[Document(id='d71fe0d6-8fe1-47a8-8ef9-a9252f53d10d', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy thier own space'),
 Document(id='cfd0da9f-509b-474e-a892-f3dbbf20b535', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and frienliness'),
 Document(id='a2fba3e6-11eb-42d4-a04d-9ee2fb32a4c4', metadata={'source': 'mammal-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(id='f51671f9-be0f-4a34-89af-15b839802725', metadata={'source': 'mammal-pets-doc'}, page_content='Goldfish are populer pets for beginners, requiring relatively simple care.')]

In [13]:
## Async query
await vectorstore.asimilarity_search("cat")

[Document(id='d71fe0d6-8fe1-47a8-8ef9-a9252f53d10d', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy thier own space'),
 Document(id='cfd0da9f-509b-474e-a892-f3dbbf20b535', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and frienliness'),
 Document(id='a2fba3e6-11eb-42d4-a04d-9ee2fb32a4c4', metadata={'source': 'mammal-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(id='f51671f9-be0f-4a34-89af-15b839802725', metadata={'source': 'mammal-pets-doc'}, page_content='Goldfish are populer pets for beginners, requiring relatively simple care.')]

In [14]:
vectorstore.similarity_search_with_score("cat")

[(Document(id='d71fe0d6-8fe1-47a8-8ef9-a9252f53d10d', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy thier own space'),
  0.9321364760398865),
 (Document(id='cfd0da9f-509b-474e-a892-f3dbbf20b535', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and frienliness'),
  1.5110875368118286),
 (Document(id='a2fba3e6-11eb-42d4-a04d-9ee2fb32a4c4', metadata={'source': 'mammal-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
  1.6657921075820923),
 (Document(id='f51671f9-be0f-4a34-89af-15b839802725', metadata={'source': 'mammal-pets-doc'}, page_content='Goldfish are populer pets for beginners, requiring relatively simple care.'),
  1.7912343740463257)]

## Retrievers

LangChain VectorStore objects do not subclass Runnable, and so cannot immediately by integrated into langChain Expression Langauge chains.

LangChain Retrievers are Runnables, so they impement a stardard set of methods (eg. synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.


In [15]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriver = RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriver.batch(["cat","dog"])



[[Document(id='d71fe0d6-8fe1-47a8-8ef9-a9252f53d10d', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy thier own space')],
 [Document(id='cfd0da9f-509b-474e-a892-f3dbbf20b535', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and frienliness')]]

Vectorstores implement as as_retrivar method that will generate a Retriver, specifically a VectorStoreRetriver. These retrivers include specific_search_type and search_kwargs attributes that identify what methods of the underlying vector store to call , and how to parameterize them. for instance, we can replicte the above with the following :

In [16]:
retriver=vectorstore.as_retriever(
    search_types="similarity",
    search_kwargs={"k":1}
)
retriver.batch(["cat","dog"])

[[Document(id='d71fe0d6-8fe1-47a8-8ef9-a9252f53d10d', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy thier own space')],
 [Document(id='cfd0da9f-509b-474e-a892-f3dbbf20b535', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and frienliness')]]

## Integreate retriver with chain

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.RunnablePassthrough

{question}

Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([("human",message)])

rag_chain={"context":retriver,"question":RunnablePassthrough()}|prompt|llm

rag_chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000018011DE19A0>, search_kwargs={'k': 1}),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nAnswer this question using the provided context only.RunnablePassthrough\n\n{question}\n\nContext:\n{context}\n'), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000018011D0DB20>, async_client=<groq.resources.chat.completion

In [19]:
response = rag_chain.invoke("tell me about dogs?")
print(response)

content='Dogs are great companions, known for their loyalty and friendliness.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 111, 'total_tokens': 126, 'completion_time': 0.016190288, 'completion_tokens_details': None, 'prompt_time': 0.009473747, 'prompt_tokens_details': None, 'queue_time': 0.049629833, 'total_time': 0.025664035}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_ff2b098aaf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019be1c3-1863-7730-8ab2-ee60bd2a9fb8-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 111, 'output_tokens': 15, 'total_tokens': 126}
